# TabM models for an unbalanced perpetual-futures panel

The shared TabM request resolves regression or classification explicitly. Classification requests
also resolve their continuous return target, task metrics, fold-specific class weights, and every
epoch checkpoint before fitting. The complete checkpoint catalog is the notebook output.

**Learning objectives**

- distinguish regression, binary classification, and multiclass requests;
- inspect the continuous return target and fold-specific imbalance treatment; and
- verify fitted-state and checkpoint lineage after GPU execution.

**Book reference:** Chapter 18, deep learning for tabular data.

**Prerequisites:** finalized crypto labels, features, and purged walk-forward folds; CUDA for the
canonical run.

In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import (
    ALL_LABELS,
    declared_contracts,
    freeze_official_model_population,
    model_request_catalog,
    open_study,
    plan_model_catalog,
    plan_specs,
    run_model_plan,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABELS = ALL_LABELS
PREVIEW_REDUCTIONS = {}
OVERRIDES = {"class_weight": "balanced", "device": "cuda"}

## Resolve targets, imbalance policy, and checkpoints

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
official_population = (
    freeze_official_model_population(study) if EXECUTION_TIER == "canonical" else None
)
requests = model_request_catalog("tabular_dl", labels=LABELS, config_prefix="tabm")
requests

family,label,config_name
str,str,str
"""tabular_dl""","""fwd_ret_8h""","""tabm_s"""
"""tabular_dl""","""fwd_ret_8h""","""tabm_m"""
"""tabular_dl""","""fwd_ret_8h""","""tabm_l"""
"""tabular_dl""","""fwd_ret_24h""","""tabm_s"""
"""tabular_dl""","""fwd_ret_24h""","""tabm_m"""
…,…,…
"""tabular_dl""","""fwd_dir_8h""","""tabm_m"""
"""tabular_dl""","""fwd_dir_8h""","""tabm_l"""
"""tabular_dl""","""fwd_dir_8h_3c""","""tabm_s"""


In [4]:
plan = plan_model_catalog(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides=OVERRIDES,
    preview_reductions=PREVIEW_REDUCTIONS,
)
# Task semantics and imbalance treatment are resolved inputs, so read them from the frozen
# specification rather than restating the configuration file here.
resolved_tasks = [spec.get("computation", spec)["task"] for spec in plan_specs(plan)]
resolved_contracts = declared_contracts(plan).with_columns(
    pl.Series("metrics", [task.get("metrics", []) for task in resolved_tasks]),
    pl.Series("imbalance", [task.get("imbalance") for task in resolved_tasks]),
)
resolved_contracts.select(
    "label",
    "config_name",
    "task",
    "continuous_eval_label",
    "imbalance",
    "metrics",
    "checkpoint_value",
    "eligible_rows",
    "training_hash",
)

label,config_name,task,continuous_eval_label,imbalance,metrics,checkpoint_value,eligible_rows,training_hash
str,str,str,str,struct[2],list[str],i64,i64,str
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],25,35280,"""b3a627bdc0b5"""
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],50,35280,"""b3a627bdc0b5"""
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],75,35280,"""b3a627bdc0b5"""
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],100,35280,"""b3a627bdc0b5"""
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],125,35280,"""b3a627bdc0b5"""
…,…,…,…,…,…,…,…,…
"""fwd_dir_8h_3c""","""tabm_l""","""classification""","""fwd_ret_8h""","{{[0.902901, 1.210376, 0.937849],[0.921834, 1.271327, 0.886033]},""balanced""}","[""ic"", ""accuracy"", ""balanced_accuracy""]",100,35280,"""3259e5045b00"""
"""fwd_dir_8h_3c""","""tabm_l""","""classification""","""fwd_ret_8h""","{{[0.902901, 1.210376, 0.937849],[0.921834, 1.271327, 0.886033]},""balanced""}","[""ic"", ""accuracy"", ""balanced_accuracy""]",125,35280,"""3259e5045b00"""
"""fwd_dir_8h_3c""","""tabm_l""","""classification""","""fwd_ret_8h""","{{[0.902901, 1.210376, 0.937849],[0.921834, 1.271327, 0.886033]},""balanced""}","[""ic"", ""accuracy"", ""balanced_accuracy""]",150,35280,"""3259e5045b00"""


The complete case-wide population is recorded before the first fit, so a member that later
fails to train cannot quietly disappear from the population it was declared in. This notebook
produces one slice of it, and that slice must lie inside the declaration.

In [5]:
if official_population is not None:
    outside = set(plan.expected_prediction_hashes) - set(official_population.members)
    if outside:
        raise RuntimeError(
            f"{len(outside)} declared checkpoints lie outside the official model population"
        )

## Execute and validate the fitted-state population

In [6]:
execution = run_model_plan(
    plan,
    population_name="crypto-tabm-validation-predictions-v1"
    if EXECUTION_TIER == "canonical"
    else None,
)
catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if (
    catalog.height != len(plan.expected_prediction_hashes)
    or catalog.filter(~pl.col("complete")).height
):
    raise RuntimeError("TabM fitted-state or prediction population is incomplete")
catalog.select(
    "label",
    "config_name",
    "task",
    "checkpoint_kind",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
    "complete",
)

Preparing and releasing folds...
  Fold 0: train=31,402  val=18,542


      epoch  25/200: loss=0.001528, IC=+0.0148


      epoch  50/200: loss=0.001452, IC=+0.0030


      epoch  75/200: loss=0.001396, IC=-0.0029


      epoch 100/200: loss=0.001353, IC=-0.0027


...

    → best_epoch=50, IC=+0.0379 (16.5s)


    → best_epoch=25, IC=+0.0356 (20.3s)


    → best_epoch=25, IC=+0.0311 (30.2s)



  Best: 56613697f4f1 @ epoch 50 (IC=+0.0379)


label,config_name,task,checkpoint_kind,checkpoint_value,training_hash,prediction_hash,complete
str,str,str,str,i64,str,str,bool
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",25,"""320ca8909e63""","""da82f4597ff4""",true
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",50,"""320ca8909e63""","""b73e005be45a""",true
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",75,"""320ca8909e63""","""c0243640b671""",true
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",100,"""320ca8909e63""","""d6bc890a959e""",true
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",125,"""320ca8909e63""","""8a8eff7df9fe""",true
…,…,…,…,…,…,…,…
"""fwd_ret_8h""","""tabm_s""","""regression""","""epoch""",100,"""b3a627bdc0b5""","""bdd596f5a37f""",true
"""fwd_ret_8h""","""tabm_s""","""regression""","""epoch""",125,"""b3a627bdc0b5""","""6f078b8199b3""",true
"""fwd_ret_8h""","""tabm_s""","""regression""","""epoch""",150,"""b3a627bdc0b5""","""55e22f9a8288""",true


## Key takeaways and limitations

- Task semantics and imbalance treatment are resolved inputs, not notebook-side conventions.
- Every reported checkpoint has a persisted fitted state and exact prediction coverage.
- GPU kernels can introduce small numerical differences; catalog identity still binds the model,
  seed, device policy, and checkpoint schedule used by the run.